# 🛡️ ShieldBot — Notebook 1: Prepare the Documents (Ingestion)

**Ingestion** means reading the knowledge base, cutting it into chunks, turning the chunks into numbers (embeddings), and saving them in a vector database. Run this notebook once, and again whenever the documents change. **Notebook 2 (the chatbot) only reads what this notebook builds.**

The code lives in `src/shieldbot/ingest.py`; this notebook runs it step by step and shows the evidence.

| Problem with a basic RAG | What it means here | Fix | Section |
|---|---|---|---|
| One bad file stops everything | A non-UTF-8 file or broken front matter crashes the whole run | Skip the bad file and report it | 2 |
| Metadata is ignored | `status` and `audience` are the only way to hide old and confidential docs | Parse YAML front matter into metadata (R1) | 2 |
| Divider lines are noise | `---` lines add nothing to a chunk | Clean the text | 3 |
| Tables are cut in half | The second half of a premium table has numbers but no column names | Chunk by heading, keep every table whole (R2) | 4 |
| Chunks lose their subject | "Waiting period: 90 days" — for health or life? | Context header with title, product and section | 4 |
| Embeddings cost money | Broken chunks are expensive to find later | Check chunks before embedding | 5 |
| A failed rebuild deletes all data | If OpenAI fails mid-build, nothing is left | Build in a new folder beside the old one | 6 |
| Rebuilding breaks the running chatbot | The chatbot reads a folder that is being replaced | Switch with a manifest (`index_info.json`) only when ready | 7 |
| The embedding model is written twice | Ingestion and chatbot must use the same model | Save it in the manifest | 7 |

**Run the cells from top to bottom.**

## 1. Setup

`autoreload` picks up edits to the `.py` files without restarting the kernel. All settings live in `src/shieldbot/config.py`, so we never search the code for them.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import shutil
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.manifold import TSNE

from shieldbot import config, ingest

print("Project root    :", config.ROOT)
print("Knowledge base  :", config.KB_DIR.exists())
print("API key loaded  :", bool(os.getenv("OPENAI_API_KEY")))
print("Embedding model :", config.EMBEDDING_MODEL)
print("Chunk size      :", config.CHUNK_SIZE, "characters (tables are never cut)")

Project root    : D:\shieldbot
Knowledge base  : True
API key loaded  : True
Embedding model : text-embedding-3-large
Chunk size      : 1000 characters (tables are never cut)


## 2. Load the files safely (R1)

**Problem:** one bad file (not UTF-8, or broken YAML) would stop the whole run, and a loose file in the folder could crash code that expects only folders.

**Fix:**
- `rglob("*.md")` asks only for Markdown files, so anything else is ignored.
- Each file is read inside `try` / `except`. A bad file is noted and skipped.

**Metadata** = labels stored with the text. Every document starts with a **YAML front matter** block (`doc_id`, `product`, `status`, `audience`, …). We parse it, because the chatbot uses `status` to hide superseded docs (R3) and `audience` to hide internal ones (R4).

In [2]:
documents, skipped = ingest.load_documents(config.KB_DIR)

print(f"Loaded {len(documents)} documents")
print(f"Skipped {len(skipped)} files: {skipped}")

meta_df = ingest.metadata_table(documents)
display(meta_df)

Loaded 26 documents
Skipped 0 files: []


,folder,source_file,doc_id,title,product,doc_type,effective_date,status,audience,chars,version,path,expiry_date,superseded_by,supersedes,confidentiality
0,00_company,branch_directory.md,CS-COM-002,Branch Directory,general,branch_info,2026-06-01,active,public,1397,2026.2,00_company/branch_directory.md,NaN,NaN,NaN,NaN
1,00_company,company_overview.md,CS-COM-001,Company Overview,general,company_info,2026-01-01,active,public,1848,2026.1,00_company/company_overview.md,NaN,NaN,NaN,NaN
2,00_company,contact_and_service_hours.md,CS-COM-003,Contact Channels and Service Hours,general,contact_info,2026-01-01,active,public,1029,2026.1,00_company/contact_and_service_hours.md,NaN,NaN,NaN,NaN
3,01_motor,motor_premium_table_2026.md,CS-MOT-003,Motor Premium Rates 2026,motor,premium_table,2026-01-01,active,public,1421,2026.1,01_motor/motor_premium_table_2026.md,NaN,NaN,NaN,NaN
4,01_motor,motorshield_comprehensive_policy_2026.md,CS-MOT-001,MotorShield Comprehensive — Policy Terms,motor,policy_terms,2026-01-01,active,public,3251,2026.1,01_motor/motorshield_comprehensive_policy_2026.md,NaN,NaN,NaN,NaN
5,01_motor,motorshield_third_party_policy_2026.md,CS-MOT-002,MotorShield Third Party — Policy Terms,motor,policy_terms,2026-01-01,active,public,1098,2026.1,01_motor/motorshield_third_party_policy_2026.md,NaN,NaN,NaN,NaN
6,02_health,health_premium_table_2026.md,CS-HLT-003,Health Shield Plus Premium Table 2026,health,premium_table,2026-01-01,active,public,787,2026.1,02_health/health_premium_table_2026.md,NaN,NaN,NaN,NaN
7,02_health,health_shield_plus_policy_2025.md,CS-HLT-001,Health Shield Plus — Policy Terms (2025),health,policy_terms,2025-01-01,superseded,public,1239,2025.1,02_health/health_shield_plus_policy_2025.md,2025-12-31,CS-HLT-002,NaN,NaN
8,02_health,health_shield_plus_policy_2026.md,CS-HLT-002,Health Shield Plus — Policy Terms (2026),health,policy_terms,2026-01-01,active,public,2607,2026.1,02_health/health_shield_plus_policy_2026.md,NaN,NaN,CS-HLT-001,NaN
9,03_life,life_premium_table.md,CS-LIF-002,Life Secure Premium Table,life,premium_table,2026-01-01,active,public,1026,2026.1,03_life/life_premium_table.md,NaN,NaN,NaN,NaN


In [3]:
print("By status:  ", meta_df["status"].value_counts().to_dict())
print("By audience:", meta_df["audience"].value_counts().to_dict())
print("By product: ", meta_df["product"].value_counts().to_dict())
print("Superseded: ", meta_df.loc[meta_df["status"] == "superseded", "source_file"].tolist())
print("Internal:   ", meta_df.loc[meta_df["audience"] == "internal", "source_file"].tolist(), "\n")

ingest.check_documents(documents)

By status:   {'active': 25, 'superseded': 1}
By audience: {'public': 24, 'internal': 2}
By product:  {'general': 12, 'motor': 4, 'health': 4, 'life': 3, 'travel': 2, 'home': 1}
Superseded:  ['health_shield_plus_policy_2025.md']
Internal:    ['agent_handbook.md', 'fraud_awareness_guidelines.md'] 

Document checks passed: 26 docs, 1 superseded, 2 internal, no missing fields


**Prove it works.** This cell makes a small test folder with one good file, one non-UTF-8 file, one file without front matter and one loose `.txt` file. A basic loader would crash; ours loads the good file and reports the rest.

In [4]:
test_folder = Path(tempfile.mkdtemp())
(test_folder / "01_motor").mkdir()

(test_folder / "01_motor" / "good.md").write_text(
    "---\ndoc_id: T1\ntitle: Test\nproduct: motor\nstatus: active\naudience: public\n---\n# Test\nHello",
    encoding="utf-8")
(test_folder / "01_motor" / "bad.md").write_bytes("---\ndoc_id: T2\n---\nCafé".encode("cp1252"))  # not UTF-8
(test_folder / "01_motor" / "no_meta.md").write_text("# No front matter here", encoding="utf-8")
(test_folder / "notes.txt").write_text("A loose file")

test_documents, test_skipped = ingest.load_documents(test_folder)
print("Loaded: ", [d.metadata["source_file"] for d in test_documents])
print("Skipped:", test_skipped)

shutil.rmtree(test_folder)

Loaded:  ['good.md']
Skipped: ['bad.md (not UTF-8)', 'no_meta.md (no front matter)']


## 3. Clean the text

**Problem:** horizontal divider lines (`---`) and long blank gaps add noise to chunks and pull the search toward empty text.

**Fix:** remove them with `re.sub` ("find this pattern and replace it"). Table separator rows like `|---|---|` start with `|`, so they are **kept**.

| Pattern | Meaning |
|---|---|
| `_{5,}` | 5 or more underscores (signature lines) |
| `^---\s*$` | a line that only contains `---` |
| `\n{3,}` | 3 or more line breaks in a row |

In [5]:
import re

divider = re.compile(r"^\s*-{3,}\s*$", re.MULTILINE)
print("Divider lines before cleaning:", sum(len(divider.findall(d.page_content)) for d in documents))
print("Divider lines after cleaning: ", sum(len(divider.findall(ingest.clean_text(d.page_content))) for d in documents))

Divider lines before cleaning: 0
Divider lines after cleaning:  0


## 4. Smart chunks (R2)

This is the most important fix in this notebook.

**Problem 1 — tables cut in half.** Premiums, waiting periods, branch hours and refunds live in Markdown tables. A 500-character cut (our Step 2 baseline) splits a table, so the second half has numbers but no column names.

**Problem 2 — chunks lose their subject.** "Waiting period" appears in health *and* life documents with different values. A chunk that only says `| Pre-existing | 12 months |` does not say which product it belongs to.

**Fix, in three parts:**

1. **Heading-based chunking.** `MarkdownHeaderTextSplitter` cuts at `#`, `##` and `###`, so each section stays whole. Headings differ between documents (`## 3. Plans`, `## Section 4 — Notification Time Limits`), so we split on any heading, never on a numbering pattern.
2. **Tables are never cut.** Inside a section, every run of lines starting with `|` is one block. A long section is packed into pieces **between** blocks only. A later piece repeats the heading with `(continued)`, so a table always travels with its heading.
3. **Context header.** Every chunk starts with a label:

```
Document: Health Shield Plus Policy (2026 Edition)
Product: health | Type: policy_terms | Effective: 2026-01-01
Section: Health Shield Plus > 4. Waiting Periods
```

The header is embedded **with** the text, so the word *health* now sits next to the waiting-period table, and vector search, BM25 and the reranker can all tell it apart from the life one.

**Why we keep superseded and internal docs in the index:** the chatbot filters them **at search time** (Notebook 2). Staff mode can then reach internal documents without a second database.

In [6]:
chunks = ingest.chunk_documents(documents)
print(f"{len(documents)} documents became {len(chunks)} chunks")

chunk_df = pd.DataFrame([{**c.metadata, "chars": len(c.page_content)} for c in chunks])
display(chunk_df.groupby("source_file").agg(chunks=("chunk_id", "count"),
                                            tables=("has_table", "sum"),
                                            largest=("chars", "max")))

26 documents became 128 chunks


,chunks,tables,largest
source_file,,,
agent_handbook.md,5,2,610
branch_directory.md,3,1,1086
claim_documents_checklist.md,2,1,1115
claims_overview_procedure.md,8,1,1084
company_overview.md,5,1,769
complaints_and_grievance_procedure.md,5,1,571
contact_and_service_hours.md,4,2,545
faq_general.md,10,0,437
fraud_awareness_guidelines.md,4,2,913


**Look at the result.** The waiting-period chunk of the 2026 health policy: the whole table, its header row, its heading, and a context header that names the product.

In [7]:
for chunk in chunks:
    m = chunk.metadata
    if m["source_file"] == "health_shield_plus_policy_2026.md" and "waiting" in m["section"].lower():
        print(chunk.page_content)
        print("\n" + "=" * 80 + "\n")

Document: Health Shield Plus — Policy Terms (2026)
Product: health | Type: policy_terms | Effective: 2026-01-01
Section: Health Shield Plus — Policy Terms (2026 Edition) > 4. Waiting Periods

## 4. Waiting Periods

| Type | Waiting Period |
|---|---|
| General illness | 30 days from policy start |
| Accidents | No waiting period — covered from day 1 |
| Pre-existing conditions | 12 months |
| Maternity (Gold and Platinum only) | 9 months |
| Specified surgeries (cataract, hernia, knee replacement, tonsillectomy) | 6 months |




## 5. Check the chunks before we pay for embeddings

Embeddings cost money, so we check first. The key check: **every chunk that holds table rows must also hold the table's header row** (`|---|`). If a table had been cut, the second half would fail this test.

In [8]:
display(ingest.check_chunks(chunks))

Chunk checks passed: unique ids, every table chunk has its header row


,chunks,with_table,smallest,median,largest
0,128,42,251,434,1115


## 6. Build the new database safely

**Problem:** deleting the old database first and then building the new one means that if OpenAI fails in between, all the data is gone. It is like demolishing your house before the new one is built.

**Fix:** every build goes into its **own new folder** under `storage/`, named with the date and time. The old folder is never touched here. If this cell fails, the old database is still there and the chatbot keeps working.

**Note:** this cell calls the OpenAI embeddings API (`text-embedding-3-large`), so it costs a small amount (well under one US cent for this knowledge base) each time you run it.

In [9]:
build_folder, vectorstore = ingest.build_index(chunks)
print(f"Built {build_folder.name}")

Built build_20261126_090450


## 7. Switch to the new database

One small file fixes two problems:

**Problem 1:** if the embedding model is written in both notebooks and they disagree, the chatbot's question vectors do not match the stored chunk vectors.

**Problem 2:** rebuilding while the chatbot is running can break it.

**Fix:** `storage/index_info.json` — a **manifest** — records which build is current and which embedding model made it. The chatbot reads it, so it always uses the right model and notices a new build.

**Safety check:** we only switch if every chunk was saved. We write a `.tmp` file first and then rename it, because a rename happens all at once: the chatbot can never read a half-written manifest.

In [10]:
import json

index_info = ingest.switch_to_build(vectorstore, build_folder, expected_count=len(chunks))
print(json.dumps(index_info, indent=2))

{
  "index_path": "storage/build_20261126_090450",
  "embedding_model": "text-embedding-3-large",
  "collection_name": "ceylon_shield",
  "chunk_count": 128,
  "built_at": "2026-11-26T09:05:33"
}


Now we delete old builds, so they do not fill up the disk. We keep the previous one as a backup, because a running chatbot may still be using it. `ignore_errors=True` means that if Windows is still using a folder, it is simply left for next time.

In [11]:
found, removed = ingest.remove_old_builds()
print(f"Builds found: {found}. Removed: {removed}.")

Builds found: 1. Removed: 0.


## 8. Test the search

Before moving on, we ask the database the **version-trap question** (Q01) twice:

1. **No filter** — what a basic RAG sees. The superseded 2025 policy competes with the 2026 one.
2. **Customer filter** (`status = active` **and** `audience = public`) — what Notebook 2 will use. The 2025 policy and internal documents can no longer appear.

In [12]:
question = "What is the waiting period for pre-existing conditions on Health Shield Plus?"
customer_filter = {"$and": [{"status": "active"}, {"audience": "public"}]}

def show(results):
    for number, doc in enumerate(results, start=1):
        m = doc.metadata
        print(f"  {number}. {m['source_file']:42} {m['section'][:35]:35} status={m['status']:10} audience={m['audience']}")

print("No filter:")
show(vectorstore.similarity_search(question, k=4))
print("\nCustomer filter (active + public):")
show(vectorstore.similarity_search(question, k=4, filter=customer_filter))

No filter:
  1. health_shield_plus_policy_2025.md          3. Waiting Periods                  status=superseded audience=public
  2. health_shield_plus_policy_2026.md          4. Waiting Periods                  status=active     audience=public
  3. health_shield_plus_policy_2026.md          2. What's New in 2026               status=active     audience=public
  4. health_shield_plus_policy_2025.md          Health Shield Plus — Policy Terms ( status=superseded audience=public

Customer filter (active + public):
  1. health_shield_plus_policy_2026.md          4. Waiting Periods                  status=active     audience=public
  2. health_shield_plus_policy_2026.md          2. What's New in 2026               status=active     audience=public
  3. health_shield_plus_policy_2026.md          Health Shield Plus — Policy Terms ( status=active     audience=public
  4. health_shield_plus_policy_2026.md          1. Overview                         status=active     audience=public


In [13]:
# Confidential check: in Customer mode, an agent-commission question must find no internal chunk
results = vectorstore.similarity_search("What commission do agents earn on life policies?", k=10, filter=customer_filter)
assert all(d.metadata["audience"] == "public" for d in results)
assert all(d.metadata["status"] == "active" for d in results)
print("Customer filter blocks internal and superseded chunks")

Customer filter blocks internal and superseded chunks


## 9. See the chunks on a chart

Each embedding has 3,072 numbers, which we cannot draw. **t-SNE** squeezes them down to 2 numbers, so similar chunks stay close together. Colours are chosen automatically for every product, so a new product never breaks the chart.

Look for: products forming their own groups, and the superseded 2025 health chunks (their own marker symbol) sitting right on top of the 2026 ones — that is exactly why the version filter is needed.

In [14]:
data = vectorstore.get(include=["embeddings", "metadatas"])
vectors = np.array(data["embeddings"])
points = TSNE(n_components=2, random_state=42, perplexity=min(30, len(vectors) - 1)).fit_transform(vectors)

figure = px.scatter(
    x=points[:, 0],
    y=points[:, 1],
    color=[m["product"] for m in data["metadatas"]],
    symbol=[m["status"] if m["audience"] == "public" else "internal" for m in data["metadatas"]],
    hover_name=[f"{m['title']} — {m['section']}" for m in data["metadatas"]],
    title="ShieldBot chunks, coloured by product",
)
figure.show()